### ingest_daily_support_tickets

In [2]:
pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/9.9 MB 18.7 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.9 MB 15.5 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.9 MB 11.7 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/9.9 MB 10.1 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.9 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 9.4 MB/s  0:00:01
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.4 MB 8.9 MB/s eta 0:00:02
   ------------ --------------------------- 3.9/12.4 MB 10.0 MB/s eta 0:00:01
   ------------------ --------------------- 5.8/12.4 MB 9.6 MB/s eta 0:00:01
   --------------------------- ------------ 8.7/12.4 MB 10.4 MB/s eta 0:00:01
   ------------------------------------- -- 11.8/12.4 MB 11.2 MB/s eta 0:00:01
   ------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
pip install boto3

   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ------ --------------------------------- 2.4/14.9 MB 12.4 MB/s eta 0:00:02
   ------------- -------------------------- 5.0/14.9 MB 12.8 MB/s eta 0:00:01
   --------------------- ------------------ 8.1/14.9 MB 13.4 MB/s eta 0:00:01
   -------------------------- ------------- 9.7/14.9 MB 11.8 MB/s eta 0:00:01
   ------------------------------- -------- 11.8/14.9 MB 11.4 MB/s eta 0:00:01
   ------------------------------------- -- 13.9/14.9 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 14.9/14.9 MB 10.5 MB/s  0:00:01

   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ----------------------------- 1/4 [botocore]
   ---------- ---------------

In [6]:
pip install sqlalchemy

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------- ----- 1.8/2.1 MB 20.3 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 13.8 MB/s  0:00:00

   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2

In [8]:
pip install dotenv


   ---------------------------------------- 2/2 [dotenv]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine
from datetime import datetime, timedelta

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# ---------- CONFIG ----------
db_config = {
    "host": "localhost",
    "port": "3306",
    "user": "root",  # change
    "password": "root", # change
    "database": "careplus_support_db"
}

S3_BUCKET = "careplus-datastore-sarang" 
S3_PREFIX = "support-tickets/raw/"  
DATE_TRACKER_FILE = "date_tracker.txt"

import os

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [15]:
pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [9]:
# ---------- UTILITY FUNCTIONS ----------
def get_engine(config):
    return create_engine(f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

def upload_to_s3(df, bucket, key):
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client('s3', **AWS_CONFIG)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"✅ Uploaded to s3://{bucket}/{key}")

def read_last_date(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()
    return "2025-06-30"  # Starting point before 1st July

def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)

def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")

# ---------- MAIN INGESTION LOGIC ----------
def run_ingestion():
    engine = get_engine(db_config)
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Query only that day’s data
    query = f"""
        SELECT * FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """
    df = pd.read_sql(query, engine)
    print(df.shape)
    print(df.head())

    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Upload to S3
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update date tracker
    update_last_date(DATE_TRACKER_FILE, next_date)
    print(f"📅 Updated tracker to {next_date}")

# Run
if __name__ == "__main__":
    run_ingestion()

(24, 10)
    ticket_id        created_at       resolved_at  agent priority  \
0  TCK0707000  2025-07-07 01:18  2025-07-08 07:16  Rohit    Medum   
1  TCK0707001  2025-07-07 01:49               NaN  Arjun   Medium   
2  TCK0707001  2025-07-07 01:49               NaN  Arjun   Medium   
3  TCK0707002  2025-07-07 01:51  2025-07-07 19:43  Sneha       Lw   
4  TCK0707003  2025-07-07 02:40  2025-07-08 08:31  Sneha     High   

  num_interactions         IssUeCat   channel    status agent_feedback  
0                4      Login Issue     Phone  Resolved                 
1          -999999  Payment Failure     Email      Open                 
2          -999999  Payment Failure     Email      Open                 
3                8  Payment Failure     Email  Resolved                 
4                6   Account Locked  Web Form  Resolved                 
✅ Uploaded to s3://careplus-datastore-sarang/support-tickets/raw/support_tickets_2025-07-07.csv
📅 Updated tracker to 2025-07-07
